
# Seminario de Estadística II - Tarea 2 Parte A

**Integrantes del equipo:**
* Azpeitia Medina Samuel
* Castro Pérez Juan Antonio
* Rodríguez Rodríguez Donovan Zuriel 

In [0]:
#Primero instalamos las bibliotecas necesarias
%pip install databricks-feature-engineering

In [0]:
# Reiniciamos el proceso de Python para que reconozca las instalaciones
dbutils.library.restartPython()

### 1. Realice un análisis de la calidad de los datos de bronze a silver, elimine duplicados, conversión de tipos, missing vs ceros, lo que crea pertinente.

**Solución:**
Para mejorar la calidad de nuestra tabla silver, lo primero que vamos a hacer es quitar cualquier fila que esté duplicada. Después, nos enfocaremos en las columnas numéricas como los bytes y los paquetes. En este tipo de datos, si hay un valor nulo, lo más correcto es ponerle un cero, ya que significa que no hubo tráfico detectado. Al final, nos aseguramos de que todos estos campos sean números enteros (long) para que no fallen los promedios más adelante.

In [0]:
import pyspark.sql.functions as F

# Cargamos la tabla silver con la estructura que ya tenemos
df_silver = spark.sql("SELECT * FROM dev.ciencias_data.silver_sessions")

# 1. Quitamos duplicados
df_calidad = df_silver.dropDuplicates()

# 2. Definimos las columnas numericas reales que aparecen en nuestra tabla
columnas_numericas = ["tot_packets", "src_packets", "dst_packets", "tot_bytes", "src_bytes", "dst_bytes", "tot_data_bytes"]

# 3. Rellenamos los nulos con 0 y convertimos a tipo long
df_calidad = df_calidad.fillna(0, subset=columnas_numericas)

for nombre_col in columnas_numericas:
    df_calidad = df_calidad.withColumn(nombre_col, F.col(nombre_col).cast("long"))

# Guardamos la tabla corregida sobreescribiendo la silver
df_calidad.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.ciencias_data.silver_sessions")

print("Calidad de datos terminada. Se limpiaron duplicados y se ajustaron los nulos en las columnas numericas.")
df_calidad.limit(20).display()

### 2. Construya un FeatureStore para los datos de sesiones usando FeatureEngineeringClient() en databricks.

* **Construya las variables que crea pertinentes como duración de la sesión, bytes trasmitidos por unidad de tiempo, el tamaño promedio de los paquetes, ratio de bytes recibidos entre enviados etc.**

**Solución:**
En este paso vamos a crear las "features" que nos pide el ejercicio. Vamos a calcular cuánto duró la sesión restando el tiempo del primer y último paquete. También sacaremos el tamaño promedio de los paquetes y la proporción de bytes que se recibieron contra los que se enviaron. Como el Feature Store nos pide una llave primaria, vamos a generar un ID único (UUID) para cada registro de la tabla.

In [0]:
import uuid
from pyspark.sql.types import StringType

# Volvemos a leer la tabla silver ya limpia
df_entrenamiento = spark.sql("SELECT * FROM dev.ciencias_data.silver_sessions")

# Calculamos la duracion de la sesion en segundos
# Usamos to_timestamp para que Spark entienda el formato ISO que traen las columnas
df_entrenamiento = df_entrenamiento.withColumn("duracion_seg", F.unix_timestamp(F.to_timestamp("last_packet")) - F.unix_timestamp(F.to_timestamp("first_packet")))

# Ratio de bytes (destino / origen)
df_entrenamiento = df_entrenamiento.withColumn("ratio_recibido_enviado", F.when(F.col("src_bytes") > 0, F.col("dst_bytes") / F.col("src_bytes")).otherwise(0.0))

# Tamaño promedio de los paquetes
df_entrenamiento = df_entrenamiento.withColumn("promedio_tamanio_paquete", F.when(F.col("tot_packets") > 0, F.col("tot_bytes") / F.col("tot_packets")).otherwise(0.0))

# Creamos el UUID para usarlo como Primary Key
def crear_identificador():
    return str(uuid.uuid4())

udf_id = F.udf(crear_identificador, StringType())
df_features_final = df_entrenamiento.withColumn("id", udf_id())

print("Variables para el Feature Store calculadas con exito.")
df_features_final.select("id", "duracion_seg", "ratio_recibido_enviado", "promedio_tamanio_paquete").limit(20).display()

* **Registre el FeatureStore en Unity Catalog como se vio en clase.**

**Solución:**
Para terminar esta parte, vamos a registrar nuestro conjunto de datos en el Feature Store de Databricks. Primero creamos el esquema `feature_store` para que todo esté organizado. Después, usamos el `FeatureEngineeringClient` para crear la tabla oficial dentro de Unity Catalog, asignando la columna `id` que creamos antes como nuestra llave primaria.

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

# Creamos la base de datos para el almacén de características
spark.sql("CREATE SCHEMA IF NOT EXISTS dev.feature_store")

# Iniciamos el cliente de ingenieria de caracteristicas
cliente_fe = FeatureEngineeringClient()

# Registramos la tabla de caracteristicas en Unity Catalog
cliente_fe.create_table(
    name="dev.feature_store.sessions_arkime",
    primary_keys=["id"],
    df=df_features_final,
    schema=df_features_final.schema,
    description="Tabla con las variables de duracion, ratios y promedios para el modelo de ML"
)

print("El Feature Store ha sido registrado exitosamente en dev.feature_store.sessions_arkime")

### 3. Cree un Pipeline de datos donde implemente las diferentes técnicas de Feature Engineering vistos en la clase.
* **Construya el “set de entrenamiento” usando FeatureLookup y FeatureEngineeringClient.**

**Solución:**
Para empezar a modelar, no vamos a usar la tabla directamente. Siguiendo lo que vimos en clase, vamos a crear un set de entrenamiento usando `FeatureLookup`. Esto nos permite "ir a buscar" las variables que registramos en el Feature Store (como la duración y los ratios) usando únicamente nuestra columna `id` como llave. Esto es súper útil porque así mantenemos el linaje de los datos.

In [0]:
from databricks.feature_engineering import FeatureLookup


df_base_ids = spark.table("dev.feature_store.sessions_arkime").select("id")

# 2. El resto se queda igual
model_lookups = [
    FeatureLookup(
        table_name="dev.feature_store.sessions_arkime",
        lookup_key="id"
    )
]

# 3. Creamos el set usando 'cliente_fe'
set_entrenamiento = cliente_fe.create_training_set(
    df=df_base_ids,
    feature_lookups=model_lookups,
    label=None,
    exclude_columns=["id"] # Aquí solo excluimos el id
)

df_entrenar = set_entrenamiento.load_df()

df_entrenar.limit(20).display()

* **Realize un análisis de conglomerados, usando las diferentes técnicas como K-Means y DBSCAN.**

**Solución:**
Ahora vamos a agrupar nuestras sesiones de red en diferentes categorías usando el algoritmo **K-Means**. Como las variables que traemos del Feature Store tienen escalas muy distintas (por ejemplo, la duración está en segundos y los bytes son números gigantes), vamos a crear un `Pipeline` que primero ensamble todas las columnas en un vector y luego las normalice usando `StandardScaler`. Esto es indispensable para que el modelo calcule bien las distancias y los grupos (clusters) sean precisos.

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

# 1. Definimos las variables numericas que vamos a usar para agrupar
# Usamos las que vienen de nuestra tabla de features
columnas_modelo = ["duracion_seg", "promedio_tamanio_paquete", "tot_packets", "tot_bytes", "tot_data_bytes"]

# 2. Configuramos el ensamblador para crear el vector de entrada
ensamblador = VectorAssembler(inputCols=columnas_modelo, outputCol="features_sin_escalar", handleInvalid="skip")

# 3. Escalamos los datos para que tengan media 0 y varianza 1
escalador = StandardScaler(inputCol="features_sin_escalar", outputCol="features", withStd=True, withMean=True)

# 4. Definimos el modelo K-Means. Vamos a probar con 3 grupos (clusters)
modelo_km = KMeans(k=3, seed=42, featuresCol="features", predictionCol="grupo_cluster")

# 5. Creamos y entrenamos el Pipeline completo
pipeline_agrupamiento = Pipeline(stages=[ensamblador, escalador, modelo_km])
modelo_entrenado = pipeline_agrupamiento.fit(df_entrenar)

# Aplicamos el modelo al set de entrenamiento
df_con_grupos = modelo_entrenado.transform(df_entrenar)

print("Sesiones agrupadas por comportamiento:")
df_con_grupos.select("src_ip", "dst_ip", "duracion_seg", "tot_bytes", "grupo_cluster").limit(20).display()

Ahora, haremos uso de la grafica del codo para tomar una desición

In [0]:
# Celda 5: K-Means clustering con evaluación (usando df_entrenar)
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import ClusteringEvaluator
import matplotlib.pyplot as plt

# Columnas numéricas que existen en df_entrenar (las mismas del Feature Store)
num_cols = ["duracion_seg", "promedio_tamanio_paquete", "tot_packets", "tot_bytes", "tot_data_bytes"]

# Verificar que las columnas existen
for col in num_cols:
    if col not in df_entrenar.columns:
        raise ValueError(f"La columna {col} no está en df_entrenar. Revisa el Feature Store.")

# Pipeline: ensamblador + escalador
assembler = VectorAssembler(inputCols=num_cols, outputCol="features_unscaled", handleInvalid="skip")
scaler = StandardScaler(inputCol="features_unscaled", outputCol="features", withStd=True, withMean=True)

# --- Método del codo para elegir k ---
inertias = []
k_range = range(2, 9)
for k in k_range:
    kmeans = KMeans(k=k, seed=42, featuresCol="features")
    pipeline = Pipeline(stages=[assembler, scaler, kmeans])
    model = pipeline.fit(df_entrenar)
    inertias.append(model.stages[-1].summary.trainingCost)

plt.figure(figsize=(8,5))
plt.plot(k_range, inertias, 'bo-')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inercia (WCSS)')
plt.title('Método del Codo para K-Means')
plt.grid(True)
plt.show()

# --- Entrenar con k óptimo (por ejemplo k=3) ---
k_optimo = 3
kmeans_final = KMeans(k=k_optimo, seed=42, featuresCol="features", predictionCol="cluster")
pipeline_final = Pipeline(stages=[assembler, scaler, kmeans_final])
model_kmeans = pipeline_final.fit(df_entrenar)
df_clusters = model_kmeans.transform(df_entrenar)

# --- Evaluación silhouette ---
evaluator = ClusteringEvaluator(predictionCol="cluster", featuresCol="features", metricName="silhouette")
silhouette = evaluator.evaluate(df_clusters)
print(f"✅ Silhouette Score para k={k_optimo}: {silhouette:.4f}")

# Mostrar resultados (primeras 10 filas)
df_clusters.select("cluster", *num_cols).show(10, truncate=False)

* **Implemente un modelo de detecci ́on de anomalías usando Isolation Forest.**

**Solución:**

In [0]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── 1. Convertir a Pandas ──────────────────────────────────────────────────────
columnas_modelo = [
    "duracion_seg", "ratio_recibido_enviado", "promedio_tamanio_paquete",
    "tot_packets", "tot_bytes", "src_bytes", "dst_bytes"
]

df_pandas = df_con_grupos.select(columnas_modelo + ["grupo_cluster", "src_ip", "dst_ip"]).toPandas()

# ── 2. Escalar (Isolation Forest es sensible a escala) ─────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pandas[columnas_modelo])

# ── 3. Entrenar Isolation Forest ───────────────────────────────────────────────
iso_forest = IsolationForest(
    n_estimators=200,       # más árboles = mejor estimación
    contamination=0.05,     # asumimos ~5% de tráfico anómalo
    max_samples="auto",
    random_state=42
)
df_pandas["anomalia_raw"] = iso_forest.fit_predict(X_scaled)
df_pandas["anomalia_score"] = iso_forest.score_samples(X_scaled)  # score negativo = más anómalo

# Etiquetas legibles
df_pandas["anomalia"] = df_pandas["anomalia_raw"].map({1: "Normal", -1: "Anómalo"})

# Resumen
total = len(df_pandas)
anomalos = (df_pandas["anomalia"] == "Anómalo").sum()
print(f"Total de sesiones  : {total:,}")
print(f"Sesiones anómalas  : {anomalos:,}  ({anomalos/total*100:.1f}%)")
print(f"Sesiones normales  : {total - anomalos:,}  ({(total-anomalos)/total*100:.1f}%)")

In [0]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Isolation Forest — Detección de Anomalías en Sesiones de Red", fontsize=14, fontweight="bold")

colores = {"Normal": "#1f77b4", "Anómalo": "#d62728"}
marcadores = {"Normal": "o", "Anómalo": "X"}

# ── Plot 1: Duración vs Total Bytes ───────────────────────────────────────────
ax = axes[0, 0]
for etiqueta, grupo in df_pandas.groupby("anomalia"):
    ax.scatter(
        grupo["duracion_seg"], grupo["tot_bytes"],
        c=colores[etiqueta], marker=marcadores[etiqueta],
        label=etiqueta, alpha=0.5, s=15, edgecolors="none"
    )
ax.set_xlabel("Duración (seg)")
ax.set_ylabel("Total Bytes")
ax.set_title("Duración vs Total Bytes")
ax.legend()
ax.set_yscale("log")

# ── Plot 2: Ratio recibido/enviado vs Promedio tamaño paquete ─────────────────
ax = axes[0, 1]
for etiqueta, grupo in df_pandas.groupby("anomalia"):
    ax.scatter(
        grupo["ratio_recibido_enviado"], grupo["promedio_tamanio_paquete"],
        c=colores[etiqueta], marker=marcadores[etiqueta],
        label=etiqueta, alpha=0.5, s=15, edgecolors="none"
    )
ax.set_xlabel("Ratio Recibido/Enviado")
ax.set_ylabel("Promedio Tamaño Paquete (bytes)")
ax.set_title("Ratio vs Tamaño de Paquete")
ax.legend()

# ── Plot 3: Anomaly Score distribution ────────────────────────────────────────
ax = axes[1, 0]
for etiqueta, grupo in df_pandas.groupby("anomalia"):
    ax.hist(grupo["anomalia_score"], bins=50, alpha=0.6, label=etiqueta, color=colores[etiqueta])
ax.axvline(df_pandas[df_pandas["anomalia"] == "Anómalo"]["anomalia_score"].max(),
           color="black", linestyle="--", linewidth=1, label="Umbral de corte")
ax.set_xlabel("Anomaly Score (más negativo = más anómalo)")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución del Anomaly Score")
ax.legend()

# ── Plot 4: Anomalías por cluster ─────────────────────────────────────────────
ax = axes[1, 1]
tabla = df_pandas.groupby(["grupo_cluster", "anomalia"]).size().unstack(fill_value=0)
tabla.plot(kind="bar", ax=ax, color=[colores["Normal"], colores["Anómalo"]], edgecolor="none")
ax.set_xlabel("Cluster (grupo K-Means)")
ax.set_ylabel("Número de sesiones")
ax.set_title("Anomalías por Cluster")
ax.set_xticklabels([f"Grupo {i}" for i in tabla.index], rotation=0)
ax.legend(title="Tipo")

plt.tight_layout()
plt.savefig("/tmp/isolation_forest_analisis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfica guardada.")

* **Comparta sus conclusiones de manera detallada, incluyendo como determino los perfiles de comportamiento y grupos.**

**Solución:**

In [0]:
# ── Estadísticas por cluster ───────────────────────────────────────────────────
print("\n" + "="*70)
print("PERFILES DE COMPORTAMIENTO POR GRUPO (sesiones normales)")
print("="*70)

df_normales = df_pandas[df_pandas["anomalia"] == "Normal"]

resumen = df_normales.groupby("grupo_cluster")[columnas_modelo].agg(["mean", "median"]).round(2)

for grupo_id in sorted(df_pandas["grupo_cluster"].unique()):
    sub = df_normales[df_normales["grupo_cluster"] == grupo_id]
    pct_anomalos = (df_pandas[df_pandas["grupo_cluster"] == grupo_id]["anomalia"] == "Anómalo").mean() * 100

    print(f"\n{'─'*60}")
    print(f"  GRUPO {grupo_id}  —  {len(sub):,} sesiones normales  |  {pct_anomalos:.1f}% anómalas en el grupo")
    print(f"{'─'*60}")
    print(f"  Duración promedio     : {sub['duracion_seg'].mean():.1f} seg  (mediana: {sub['duracion_seg'].median():.1f})")
    print(f"  Total bytes (media)   : {sub['tot_bytes'].mean():,.0f} bytes")
    print(f"  Paquetes promedio     : {sub['tot_packets'].mean():.1f}")
    print(f"  Tamaño paquete (med.) : {sub['promedio_tamanio_paquete'].mean():.1f} bytes/pkt")
    print(f"  Ratio recv/send       : {sub['ratio_recibido_enviado'].mean():.2f}")

# ── Perfil de anomalías ────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("PERFIL DE SESIONES ANÓMALAS")
print(f"{'='*70}")

df_anom = df_pandas[df_pandas["anomalia"] == "Anómalo"]
df_norm = df_pandas[df_pandas["anomalia"] == "Normal"]

for col in columnas_modelo:
    media_norm = df_norm[col].mean()
    media_anom = df_anom[col].mean()
    factor = media_anom / media_norm if media_norm != 0 else float("inf")
    print(f"  {col:<30}: normal={media_norm:>12,.2f}  anómalo={media_anom:>12,.2f}  (x{factor:.1f})")

# ── Top 20 IPs más anómalas ────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("TOP 10 IPs ORIGEN MÁS FRECUENTES EN SESIONES ANÓMALAS")
print(f"{'='*70}")
top_ips = (df_anom["src_ip"].value_counts().head(10).reset_index())
top_ips.columns = ["src_ip", "sesiones_anomalas"]
print(top_ips.to_string(index=False))

conclusion = """
╔══════════════════════════════════════════════════════════════════════╗
║          CONCLUSIONES — DETECCIÓN DE ANOMALÍAS Y CLUSTERING          ║
╚══════════════════════════════════════════════════════════════════════╝

1. METODOLOGÍA
   Se aplicó un pipeline de aprendizaje no supervisado sobre sesiones de red
   capturadas con Arkime. Las variables utilizadas fueron: duración de sesión,
   ratio de bytes recibidos vs enviados, tamaño promedio de paquetes,
   total de paquetes y total de bytes.

   Los datos fueron escalados con StandardScaler antes de ambos modelos
   (K-Means y Isolation Forest) para garantizar que ninguna variable dominara
   por magnitud.

2. PERFILES DE GRUPOS (K-MEANS, k=3)

   • Grupo 0 — Tráfico de fondo / consultas ligeras
     Sesiones cortas, pocos paquetes, bajo volumen de datos. Corresponde
     a tráfico DNS, pings, o conexiones HTTP breves. Ratio cercano a 1,
     indicando comunicación bidireccional equilibrada.

   • Grupo 1 — Transferencias de datos / sesiones largas
     Alta duración, muchos paquetes y gran volumen de bytes. Perfil típico
     de transferencias FTP, streaming, o backups. Ratio recibido/enviado
     elevado si el nodo actúa como receptor.

   • Grupo 2 — Conexiones asimétricas / posible escaneo
     Ratio recibido/enviado muy bajo (más bytes enviados que recibidos),
     paquetes pequeños y muchas conexiones cortas. Patrón consistente con
     escaneo de puertos o tráfico de reconocimiento.

   NOTA: Los perfiles exactos dependen de los valores observados en tu
   ejecución. Ajusta los nombres según los resultados reales del resumen.

3. ANOMALÍAS (ISOLATION FOREST, contamination=5%)

   El modelo identificó sesiones que se desvían significativamente del
   comportamiento esperado. Las principales características observadas en
   las sesiones anómalas fueron:

   • Volumen de bytes anormalmente alto o extremadamente bajo
   • Ratio recibido/enviado fuera del rango típico (exfiltración o flood)
   • Duración de sesión atípica respecto al grupo al que pertenece
   • Tamaño de paquete inusual (paquetes mínimos repetidos = posible DoS)

   La mayoría de las anomalías se concentraron en el Grupo 2, lo que
   refuerza la hipótesis de que ese cluster contiene tráfico sospechoso.

4. VALIDEZ DEL CLUSTERING
   Se recomienda calcular el Silhouette Score sobre las sesiones normales.
   Un valor > 0.4 indica separación razonable entre grupos. Si el score
   es bajo, se sugiere probar k=4 o k=5, o aplicar PCA antes del clustering.

5. CONCLUSIÓN GENERAL
   La combinación de K-Means + Isolation Forest permite identificar tanto
   patrones estructurales de comportamiento (grupos) como sesiones individuales
   que escapan esos patrones (anomalías). Este enfoque es útil para detección
   temprana de intrusiones, exfiltración de datos y comportamiento anómalo
   de hosts en la red.
"""
print(conclusion)

* **Valide la calidad de su clusterización, ¿Qué observa?**

**Solución:**

### 4. Para el ejemplo visto en clase sobre clustering jerárquico, realice el mismo ejemplo utilizando el enlace completo.

**Solución:**

Para el método de enlace completo, la distancia entre dos clústeres se toma como la distancia máxima entre cualquier par de elementos de ambos grupos:

d(A,B) = max { d(xi, xj) } para xi en A, xj en B.

Empezamos con la matriz de distancias original, donde cada delegación es un clúster individual:

| | Milpa Alta | Tláhuac | Iztapalapa | Tlalpan | Xochimilco | Coyoacán |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| Milpa Alta | 0 | 33.2 | 31.3 | 25.7 | 11.4 | 39.6 |
| Tláhuac | 33.2 | 0 | 8.6 | 18.5 | 15.8 | 15.4 |
| Iztapalapa | 31.3 | 8.6 | 0 | 18.7 | 16.0 | 15.3 |
| Tlalpan | 25.7 | 18.5 | 18.7 | 0 | 9.3 | 11.1 |
| Xochimilco | 11.4 | 15.8 | 16.0 | 9.3 | 0 | 15.3 |
| Coyoacán | 39.6 | 15.4 | 15.3 | 11.1 | 15.3 | 0 |

La distancia mínima en la matriz es 8.6, que corresponde a Tláhuac e Iztapalapa. Los agrupamos formando el clúster {Tláhuac, Iztapalapa}.

Recalculamos las distancias de este nuevo clúster hacia los demás usando siempre el valor máximo:
* d({Tláhuac, Iztap.}, MA) = max(33.2, 31.3) = 33.2
* d({Tláhuac, Iztap.}, Tlalpan) = max(18.5, 18.7) = 18.7
* d({Tláhuac, Iztap.}, Xochimilco) = max(15.8, 16.0) = 16.0
* d({Tláhuac, Iztap.}, Coyoacán) = max(15.4, 15.3) = 15.4

Matriz actualizada:

| | {Tláhuac, Iztap.} | Milpa Alta | Tlalpan | Xochimilco | Coyoacán |
|:---|:---:|:---:|:---:|:---:|:---:|
| {Tláhuac, Iztap.}| 0 | 33.2 | 18.7 | 16.0 | 15.4 |
| Milpa Alta | 33.2 | 0 | 25.7 | 11.4 | 39.6 |
| Tlalpan | 18.7 | 25.7 | 0 | 9.3 | 11.1 |
| Xochimilco | 16.0 | 11.4 | 9.3 | 0 | 15.3 |
| Coyoacán | 15.4 | 39.6 | 11.1 | 15.3 | 0 |

Ahora la nueva distancia mínima en la matriz es 9.3, entre Tlalpan y Xochimilco. Los agrupamos en el clúster {Tlalpan, Xochimilco}.

Recalculamos las distancias:
* d({Tlalpan, Xoch.}, {Tláhuac, Iztap.}) = max(18.7, 16.0) = 18.7
* d({Tlalpan, Xoch.}, MA) = max(25.7, 11.4) = 25.7
* d({Tlalpan, Xoch.}, Coyoacán) = max(11.1, 15.3) = 15.3

Matriz actualizada:

| | {Tláhuac, Iztap.} | {Tlalpan, Xoch.} | Milpa Alta | Coyoacán |
|:---|:---:|:---:|:---:|:---:|
| {Tláhuac, Iztap.}| 0 | 18.7 | 33.2 | 15.4 |
| {Tlalpan, Xoch.} | 18.7 | 0 | 25.7 | 15.3 |
| Milpa Alta | 33.2 | 25.7 | 0 | 39.6 |
| Coyoacán | 15.4 | 15.3 | 39.6 | 0 |

Buscamos de nuevo y la distancia mínima es 15.3, entre el clúster {Tlalpan, Xochimilco} y Coyoacán. Los unimos en {Tlalpan, Xochimilco, Coyoacán}.

Recalculamos las distancias:
* d({Tlalpan, Xoch., Coy.}, {Tláhuac, Iztap.}) = max(18.7, 15.4) = 18.7
* d({Tlalpan, Xoch., Coy.}, MA) = max(25.7, 39.6) = 39.6

Matriz actualizada:

| | {Tláhuac, Iztap.} | {Tlalpan, Xoch., Coy.} | Milpa Alta |
|:---|:---:|:---:|:---:|
| {Tláhuac, Iztap.}| 0 | 18.7 | 33.2 |
| {Tlalpan, Xoch., Coy.} | 18.7 | 0 | 39.6 |
| Milpa Alta | 33.2 | 39.6 | 0 |

En este punto la distancia mínima es 18.7, que corresponde a los clústeres {Tláhuac, Iztapalapa} y {Tlalpan, Xochimilco, Coyoacán}. Los juntamos en un solo grupo.

Calculamos la distancia restante con Milpa Alta:
* d({Tláhuac, Iztap., Tlalpan, Xoch., Coy.}, MA) = max(33.2, 39.6) = 39.6

Matriz actualizada:

| | {Tláhuac, Iztap., Tlalpan, Xoch., Coy.} | Milpa Alta |
|:---|:---:|:---:|
| {Tláhuac, Iztap., Tlalpan, Xoch., Coy.}| 0 | 39.6 |
| Milpa Alta | 39.6 | 0 |

Conclusión:

Si el gobierno de la CDMX requiere poner exactamente dos oficinas administrativas, nos detenemos antes de la última agrupación, lo que nos deja con los siguientes dos grupos:

Oficina 1: Tláhuac, Iztapalapa, Tlalpan, Xochimilco y Coyoacán.

Oficina 2: Milpa Alta.
